In [ ]:
# Cài thư viện cho kernel hiện tại (Kaggle/Colab/local đều chạy được)
%pip install -q underthesea datasets

# Bài 2 – Phân loại văn bản (Logistic Regression)

**Mục tiêu:** xây một bộ phân loại hoàn chỉnh: dữ liệu → đặc trưng → huấn luyện → đánh giá → xem lỗi.

**Bài toán:** nhận xét của sinh viên về môn học → *tiêu cực / trung tính / tích cực* (phân tích cảm xúc).

**Khái niệm mới:**
- **Train / validation / test**: train để học, validation để chỉnh, test chỉ dùng *một lần cuối* để báo kết quả thật.
- **Logistic Regression**: học một trọng số cho mỗi từ; từ nào đẩy về nhãn nào thì trọng số cao ở nhãn đó.
- **Accuracy, Precision, Recall, F1**: các thước đo (giải thích bên dưới).
- **Pipeline**: gói các bước xử lý thành một khối để không lẫn dữ liệu train/test.

## 1. Tải dữ liệu
Bộ *Vietnamese Students' Feedback* (UIT-VSFC), tải công khai từ Hugging Face.

In [ ]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset('hkab/vietnamese_students_feedback')
train = ds['train'].to_pandas()
val   = ds['validation'].to_pandas()
test  = ds['test'].to_pandas()

TEN_NHAN = {0: 'tiêu cực', 1: 'trung tính', 2: 'tích cực'}
print(len(train), len(val), len(test))
train.head(8)

In [ ]:
# Xem mất cân bằng nhãn: một nhãn chiếm quá nhiều sẽ làm accuracy 'đẹp giả'
train['sentiment'].map(TEN_NHAN).value_counts()

**Câu hỏi 1:** Nếu một mô hình luôn đoán nhãn phổ biến nhất, accuracy là bao nhiêu? (Tính từ bảng trên.) Con số này gọi là **baseline** – mô hình của ta phải thắng nó.

*(Viết câu trả lời vào ô này.)*

## 2. Mô hình đầu tiên
Tách từ bằng `underthesea` → TF-IDF → Logistic Regression, gói trong `Pipeline`.

In [ ]:
from underthesea import word_tokenize
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

def tach_tu(s):
    return word_tokenize(s.lower(), format='text')   # 'học sinh' -> 'học_sinh'

print(tach_tu('Thầy dạy rất nhiệt tình'))

In [ ]:
import time
t = time.time()
X_train = [tach_tu(s) for s in train['sentence']]
X_val   = [tach_tu(s) for s in val['sentence']]
print(f'Tách từ xong sau {time.time()-t:.0f}s')

model = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=2)),
    ('clf',   LogisticRegression(max_iter=1000)),
])
model.fit(X_train, train['sentiment'])
print('Accuracy trên validation:', round(model.score(X_val, val['sentiment']), 3))

## 3. Đánh giá đúng cách
- **Accuracy**: tỉ lệ đoán đúng. Dễ hiểu nhưng bị lừa khi nhãn mất cân bằng.
- **Precision** (của nhãn A): trong số câu mô hình *gọi* là A, bao nhiêu thật sự là A.
- **Recall** (của nhãn A): trong số câu *thật sự* là A, mô hình tìm ra bao nhiêu.
- **F1**: trung bình điều hoà của precision và recall.

Ví dụ trong công việc: lọc hồ sơ lỗi – *recall* thấp nghĩa là bỏ sót hồ sơ lỗi; *precision* thấp nghĩa là báo nhầm quá nhiều.

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

pred = model.predict(X_val)
names = [TEN_NHAN[i] for i in range(3)]
print(classification_report(val['sentiment'], pred, target_names=names, digits=3))

ConfusionMatrixDisplay.from_predictions(val['sentiment'], pred, display_labels=names, cmap='Blues')
plt.show()

**Câu hỏi 2:** Nhãn nào có recall thấp nhất? Nhìn confusion matrix, mô hình hay nhầm nó sang nhãn nào? Vì sao theo bạn?

*(Viết câu trả lời vào ô này.)*

## 4. Mô hình học được gì? (xem trọng số)
Logistic Regression có ưu điểm lớn: giải thích được.

In [ ]:
import numpy as np
tfidf, clf = model['tfidf'], model['clf']
vocab = np.array(tfidf.get_feature_names_out())
for i, ten in TEN_NHAN.items():
    top = vocab[np.argsort(clf.coef_[i])[-10:][::-1]]
    print(f'{ten:>10}:', ', '.join(top))

## 5. Xem các câu đoán sai
Bước quan trọng nhất của người làm ML thực tế: **đọc lỗi bằng mắt**, đừng chỉ nhìn con số.

In [ ]:
sai = val.assign(du_doan=pred)
sai = sai[sai['sentiment'] != sai['du_doan']]
for _, r in sai.sample(8, random_state=0).iterrows():
    print(f"thật: {TEN_NHAN[r.sentiment]:>10} | đoán: {TEN_NHAN[r.du_doan]:>10} | {r.sentence}")

## 6. Dự đoán câu mới

In [ ]:
def du_doan(cau):
    return TEN_NHAN[model.predict([tach_tu(cau)])[0]]

for c in ['thầy dạy rất hay và nhiệt tình', 'môn học quá nặng, bài tập nhiều', 'phòng học bình thường']:
    print(du_doan(c), '<-', c)

## 7. Test set – chỉ chạy khi đã chốt mô hình
Sau khi chỉnh xong trên validation, dùng test **đúng một lần** để biết chất lượng thật.

In [ ]:
X_test = [tach_tu(s) for s in test['sentence']]
print('Accuracy test:', round(model.score(X_test, test['sentiment']), 3))

## Bài tập
1. Tính baseline ở Câu hỏi 1 bằng code, rồi so với accuracy của mô hình.
2. Đổi `ngram_range=(1, 1)` và bỏ bước tách từ (dùng `s.lower()`). Accuracy và F1 thay đổi thế nào? Tách từ có đáng công không?
3. Thử `LogisticRegression(C=...)` với C = 0.1, 1, 10, 100. C lớn = ít "phạt" hơn (dễ học thuộc). Chọn C bằng **validation**, không phải test.
4. Thử `class_weight='balanced'`. Recall của nhãn hiếm có tăng không? Precision giảm bao nhiêu?
5. Thay `LogisticRegression` bằng `LinearSVC` hoặc `MultinomialNB`. Cái nào tốt nhất?
6. **Thử thách:** chọn 3 câu bạn tự viết mà mô hình đoán sai, giải thích vì sao (phủ định, mỉa mai, từ lạ...). Đây là lý do ta cần mô hình sâu hơn ở các bài sau.